<a href="https://colab.research.google.com/github/HM-Mahibullah/ComputerVision/blob/main/digitalaccess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics

In [ ]:
import ultralytics
ultralytics.checks()

In [ ]:
!pip install roboflow

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="CxHgc1Nn3FK5vYNsBtJW")
project = rf.workspace("md-mahibullah").project("digital-access")
version = project.version(6)
dataset = version.download("yolov8")


In [ ]:
import os
import matplotlib.pyplot as plt

# আপনার ডেটাসেটের রুট পাথ
dataset_path = "/content/Digital-Access-1"

# ফোল্ডারগুলোর ভেতরের ইমেজ সংখ্যা গোনা
splits = ['train', 'valid', 'test']
image_counts = []

for split in splits:
    img_dir = os.path.join(dataset_path, split, "images")
    if os.path.exists(img_dir):
        count = len([f for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        image_counts.append(count)
    else:
        image_counts.append(0)

# বার চার্ট তৈরি করা
plt.figure(figsize=(8, 5))
bars = plt.bar(splits, image_counts, color=['#ff9999','#66b3ff','#99ff99'])

# বারের ওপরে সংখ্যা বসানো
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 2, int(yval), ha='center', va='bottom', fontweight='bold')

plt.title("Dataset Split Overview (digital-access-1)", fontsize=14, fontweight='bold')
plt.xlabel("Dataset Split", fontsize=12)
plt.ylabel("Number of Images", fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
print(dataset.location)

In [ ]:
from IPython.display import Image
Image('/content/runs/detect/train/confusion_matrix.png', width=600)

In [ ]:
Image("/content/runs/detect/train/labels.jpg", width=600)

In [ ]:
Image("/content/runs/detect/train-3/results.png", width=600)

In [ ]:
Image("/content/runs/detect/train-3/train_batch0.jpg", width=600)

In [ ]:
Image("/content/runs/detect/train-3/val_batch0_pred.jpg", width=600)

In [ ]:
!yolo task=detect mode=val model="/content/runs/detect/train-3/weights/best.pt" data={dataset.location}/data.yaml

In [ ]:
!yolo task=detect mode=predict model="/content/runs/detect/train-3/weights/best.pt" conf=0.25 source={dataset.location}/test/images save=True

In [ ]:
!yolo task=detect mode=predict model= "/content/best.pt" conf=0.25 source=image2.jpg save=True!yolo task=detect mode=predict model= "/content/best.pt" conf=0.25 source=/content/sharafat.jpg save=True

In [ ]:
import cv2
import os
import numpy as np
import base64
from PIL import Image
import io
from google.colab.output import eval_js
from IPython.display import display, Javascript
from ultralytics import YOLO

# ১. মডেল লোড
model = YOLO("/content/best.pt")

# ২. জাভাস্ক্রিপ্ট কোড: লজিটেক ক্যামেরা চালু রাখবে এবং প্রতি ফ্রেমে ডাটা পাইথনে পাঠাবে
def start_always_video():
    js = Javascript('''
        var video;
        var div;
        var stream;
        var canvas;

        async function init() {
            div = document.createElement('div');
            div.style.position = 'relative';

            video = document.createElement('video');
            video.style.display = 'block';
            video.width = 640;
            video.height = 480;

            // ওভারলে ক্যানভাস (নাম ও বক্স আঁকার জন্য)
            canvas = document.createElement('canvas');
            canvas.width = 640;
            canvas.height = 480;
            canvas.style.position = 'absolute';
            canvas.style.top = '0px';
            canvas.style.left = '0px';
            canvas.style.zIndex = '100';

            // লজিটেক এবং ব্রেভ ক্যামেরার জন্য কনস্ট্রেইন্টস
            const constraints = {
                video: { facingMode: "user", width: 640, height: 480 }
            };

            stream = await navigator.mediaDevices.getUserMedia(constraints);
            video.srcObject = stream;
            await video.play();

            document.body.appendChild(div);
            div.appendChild(video);
            div.appendChild(canvas);

            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        }

        // ক্যামেরা থেকে কারেন্ট ফ্রেম নেওয়ার ফাংশন
        async function captureFrame() {
            var tempCanvas = document.createElement('canvas');
            tempCanvas.width = 640;
            tempCanvas.height = 480;
            tempCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
            return tempCanvas.toDataURL('image/jpeg', 0.85);
        }

        // রিয়েল-টাইমে বক্স এবং নাম ড্র করার ফাংশন
        function drawBoxes(predictions) {
            var ctx = canvas.getContext('2d');
            ctx.clearRect(0, 0, 640, 480); // আগের ফ্রেমের বক্স মুছে ফেলা

            predictions.forEach(pred => {
                // উজ্জ্বল লাল রঙের বক্স
                ctx.strokeStyle = '#FF0000';
                ctx.lineWidth = 4;
                ctx.strokeRect(pred.x, pred.y, pred.w, pred.h);

                // নামের ব্যাকগ্রাউন্ড পটভূমি
                ctx.fillStyle = '#FF0000';
                ctx.font = 'bold 18px Arial';
                var textWidth = ctx.measureText(pred.name).width;
                ctx.fillRect(pred.x - 2, pred.y - 28, textWidth + 60, 28);

                // সাদা রঙে নাম এবং কনফিডেন্স লেখা
                ctx.fillStyle = '#FFFFFF';
                ctx.fillText(pred.name + ' (' + pred.conf + '%)', pred.x + 5, pred.y - 8);
            });
        }

        // ভিডিও বন্ধ করার ফাংশন
        function stopVideo() {
            stream.getVideoTracks()[0].stop();
            div.remove();
        }
    ''')
    display(js)

try:
    # ৩. ভিডিও স্ট্রিমিং ও অনবরত লুপ শুরু
    start_always_video()
    eval_js('init()')
    print("🎥 লজিটেক ক্যামেরা চালু হয়েছে... (বন্ধ করতে চাইলে বাম পাশের Stop বা জেনারেট বাটনে ক্লিক করুন)")

    while True:
        # জাভাস্ক্রিপ্ট থেকে প্রতি মুহূর্তের ফ্রেম আনা
        frame_data = eval_js('captureFrame()')
        if not frame_data:
            break

        # বেস৬৪ ইমেজকে ওপেনসিভি (OpenCV) ফ্রেমে রূপান্তর
        binary = base64.b64decode(frame_data.split(',')[1])
        image = Image.open(io.BytesIO(binary))
        frame = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)

        # YOLOv8 মডেল দিয়ে ডিটেকশন (কনফিডেন্স ০.২৫ রাখা হয়েছে)
        results = model.predict(source=frame, conf=0.005, verbose=False)

        # ডিটেকশন রেজাল্ট ফিল্টার করা
        predictions = []
        for box in results[0].boxes:
            coords = box.xyxy[0].tolist()
            x = int(coords[0])
            y = int(coords[1])
            w = int(coords[2] - coords[0])
            h = int(coords[3] - coords[1])

            conf = int(box.conf[0].item() * 100)
            cls_id = int(box.cls[0].item())
            name = model.names[cls_id]

            predictions.append({
                'x': x, 'y': y, 'w': w, 'h': h,
                'name': name, 'conf': conf
            })

        # জাভাস্ক্রিপ্ট ক্যানভাসে ডাটা পাঠানো যাতে লাইভ ভিডিওর ওপর নাম ভেসে ওঠে
        eval_js(f'drawBoxes({predictions})')

except KeyboardInterrupt:
    eval_js('stopVideo()')
    print("\n🛑 ভিডিও ক্যাপচার বন্ধ করা হয়েছে।")
except Exception as e:
    print(f"ভুল হয়েছে: {e}")
    try:
        eval_js('stopVideo()')
    except:
        pass